### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="qsar_biodeg",
    dataset_year="2013",
    domain_str="biology & life sciences",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5H60M",
    download_description="""
mkdir -p local-data-warehouse/qsar_biodeg/ && wget -P local-data-warehouse/qsar_biodeg/ https://archive.ics.uci.edu/static/public/254/qsar+biodegradation.zip && unzip local-data-warehouse/qsar_biodeg/qsar+biodegradation.zip -d local-data-warehouse/qsar_biodeg/ && rm local-data-warehouse/qsar_biodeg/qsar+biodegradation.zip
""",
    # References
    academic_reference_bibtex="""@article{mansouri2013quantitative,
  title={Quantitative structure--activity relationship models for ready biodegradability of chemicals},
  author={Mansouri, Kamel and Ringsted, Tine and Ballabio, Davide and Todeschini, Roberto and Consonni, Viviana},
  journal={Journal of chemical information and modeling},
  volume={53},
  number={4},
  pages={867--878},
  year={2013},
  publisher={ACS Publications}
}
""",
    academic_reference_bibtex_key="mansouri2013quantitative",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- We added semantic meaningful feature names.
- Anomaly: several features are numeric-ordinal in nature but it is unclear if they are categorical features.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Biodegradable",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="Biodegradable",
)

## Preprocessing

In [ ]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/biodeg.csv", sep=";")
target_feature = "Biodegradable"
df.columns  = [
    "Laplace_Leading_Eigenvalue",
    "Weighted_Balaban_Index_Barysz_Matrix",
    "Num_Heavy_Atoms",
    "Freq_NN_At_Dist1",
    "Freq_CN_At_Dist4",
    "Num_ssssC_Atoms",
    "Num_Substituted_BenzeneC",
    "Percentage_C_Atoms",
    "Num_Terminal_PrimaryC",
    "Num_Oxygen_Atoms",
    "Freq_CN_At_Dist3",
    "Sum_dssC_EStates",
    "Weighted_HyperWiener_Index_Burden_Matrix",
    "Lopping_Centric_Index",
    "Laplace_Spectral_Moment6",
    "Freq_CO_At_Dist3",
    "Mean_Sanderson_Electronegativity",
    "Mean_Ionization_Potential",
    "Num_N_Hydrazine",
    "Num_Aromatic_Nitro_Groups",
    "Num_CRX3",
    "Weighted_Normalized_SpectralPositiveSum_Burden_Matrix",
    "Num_Circuits",
    "Presence_CBr_At_Dist1",
    "Presence_CCl_At_Dist3",
    "N073_chemical_substructure", # no idea what this might be
    "Adjacency_LeadingEigenvalue",
    "Intrinsic_State_Pseudoconnectivity",
    "Presence_CBr_At_Dist4",
    "Sum_dO_EStates",
    "Laplace_MoharIndex2",
    "Num_RingTertiaryC",
    "C026_chemical_substructure",
    "Freq_CN_At_Dist2",
    "Num_HBond_Donors_Atoms",
    "Weighted_LeadingEigenvalue_Burden_Matrix",
    "Intrinsic_State_Pseudoconnectivity_SAvg",
    "Num_Nitrogen_Atoms",
    "Weighted_SpectralMoment6_Burden_Matrix",
    "Num_Esters",
    "Num_Halogen_Atoms",
    target_feature
]
df[target_feature] = df[target_feature].map({"RB": "Yes", "NRB": "No"})

cat_features = [
    target_feature,
    "Presence_CBr_At_Dist1",
    "Presence_CCl_At_Dist3",
    "N073_chemical_substructure",
    "Presence_CBr_At_Dist4",
    "C026_chemical_substructure"  # very likely categorical, but unclear
]
df[cat_features] = df[cat_features].astype("category")

# Shows a distribution shift, that means the original data was sorted
# by an order (by feature "Laplace_LeadingEigenvalue" and target). Vanishes if we
# shuffle here.
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()